# XGBoost Cross-Validation

Проверяю лучший XGBoost с 5-fold Stratified Cross-Validation

Параметры дальше выбираются по среднему ROC-AUC на train

Validation - для финальной проверки

In [2]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

SEED = 20260916

train = pd.read_csv("../train.csv")
validation = pd.read_csv("../validation.csv")

feature_columns = [
    column
    for column in train.columns
    if column not in ["row_id", "target"]
]

X_train = train[feature_columns]
y_train = train["target"]

X_validation = validation[feature_columns]
y_validation = validation["target"]

cv = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    ).split(X_train, y_train)
)

In [3]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.4,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="auc",
)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
)

cv_results = pd.DataFrame({
    "fold": range(1, 6),
    "roc_auc": scores,
})

cv_results

,fold,roc_auc
0,1,0.880606
1,2,0.896447
2,3,0.893849
3,4,0.889094
4,5,0.905313


In [4]:
print("Mean ROC-AUC:", scores.mean())
print("Std ROC-AUC:", scores.std())

Mean ROC-AUC: 0.8930618543713782
Std ROC-AUC: 0.008161749998945339


### Вывод по Cross-Validation

Средний ROC-AUC по 5 фолдам: 0.8931

Рез-ты на разных фолдах близкие

Std ROC-AUC = 0.0082

Дальше параметры XGBoost подбираю по ср. CV ROC-AUC

In [5]:
from itertools import product

cv_tuning_results = []

for n_estimators, max_depth, learning_rate in product(
    [50, 100, 200],
    [3, 5, 7],
    [0.05, 0.1, 0.2, 0.4],
):
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=1.0,
        colsample_bytree=1.0,
        min_child_weight=1,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,
    )

    cv_tuning_results.append({
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "mean_auc": scores.mean(),
        "std_auc": scores.std(),
    })

cv_tuning_df = (
    pd.DataFrame(cv_tuning_results)
    .sort_values("mean_auc", ascending=False)
    .reset_index(drop=True)
)

cv_tuning_df.head(10)

,n_estimators,max_depth,learning_rate,mean_auc,std_auc
0,200,5,0.20,0.900517,0.008171
1,200,5,0.10,0.899798,0.007246
2,200,7,0.10,0.899436,0.009246
3,200,7,0.20,0.898891,0.006702
4,200,5,0.05,0.898589,0.011109
5,200,7,0.05,0.896825,0.008555
6,200,5,0.40,0.896010,0.007675
7,100,5,0.20,0.896003,0.010117
8,200,3,0.10,0.895276,0.009198
9,100,7,0.20,0.894753,0.008070


### Вывод по coarse search

Лучший - при `n_estimators=200`, `max_depth=5`, `learning_rate=0.2`

Средний ROC-AUC вырос с 0.8931 до 0.9005

Std ROC-AUC = 0.0082

Дальше уточняю `n_estimators`

In [6]:
xgboost_estimators_cv_results = []

for n_estimators in [200, 300, 400, 500]:
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        learning_rate=0.2,
        subsample=1.0,
        colsample_bytree=1.0,
        min_child_weight=1,
        random_state=SEED,
        n_jobs=-1,
        eval_metric="auc",
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1,
    )

    xgboost_estimators_cv_results.append({
        "n_estimators": n_estimators,
        "mean_auc": scores.mean(),
        "std_auc": scores.std(),
    })

xgboost_estimators_cv_df = pd.DataFrame(
    xgboost_estimators_cv_results
)

xgboost_estimators_cv_df

,n_estimators,mean_auc,std_auc
0,200,0.900517,0.008171
1,300,0.901833,0.007463
2,400,0.902890,0.007088
3,500,0.902828,0.006969


## Вывод по n_estimators

Лучший - при `n_estimators=400`: 0.9029

До 400 деревьев средний ROC-AUC рос

При 500 улучшения уже не было

Дальше `n_estimators=400`

In [7]:
local_cv_results = []

for max_depth in [4, 5, 6]:
    for learning_rate in [0.15, 0.2, 0.25]:
        model = XGBClassifier(
            n_estimators=400,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        )

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=1,
        )

        local_cv_results.append({
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "mean_auc": scores.mean(),
            "std_auc": scores.std(),
        })

local_cv_df = (
    pd.DataFrame(local_cv_results)
    .sort_values("mean_auc", ascending=False)
    .reset_index(drop=True)
)

local_cv_df

,max_depth,learning_rate,mean_auc,std_auc
0,5,0.15,0.904283,0.006899
1,4,0.20,0.903493,0.005479
2,5,0.20,0.902890,0.007088
3,4,0.25,0.902491,0.008800
4,6,0.15,0.901934,0.006159
5,5,0.25,0.901798,0.005920
6,4,0.15,0.900819,0.007292
7,6,0.20,0.900787,0.007613
8,6,0.25,0.898951,0.005628


In [8]:
cv_xgboost = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.15,
    subsample=1.0,
    colsample_bytree=1.0,
    min_child_weight=1,
    random_state=SEED,
    n_jobs=-1,
    eval_metric="auc",
)

cv_xgboost.fit(X_train, y_train)

validation_probability = cv_xgboost.predict_proba(
    X_validation
)[:, 1]

validation_auc = roc_auc_score(
    y_validation,
    validation_probability,
)

print("Validation ROC-AUC:", validation_auc)

Validation ROC-AUC: 0.9071556122448979
